# 🍄 Taller de Clustering — Parte 1: Setas

Notebook de trabajo: solo contiene las fases ya vistas (ver [`learn_roadmap.md`](learn_roadmap.md)).

## ✅ Fase 01 · Comprensión del problema y de los datos — ya vista

> **Resumen fase 01:** 8124 setas × 23 columnas, **todas categóricas** (letras). Hay etiqueta (`class`), pero solo para validar. No hay nulos *aparentes*.

▶️ **Ejecuta primero la celda plegada de abajo** (o usa *Run All*): crea `df`, `X`, `y`… que necesita la fase actual. Pulsa `…` para ver su código.

In [1]:
%matplotlib inline
# Hace que los gráficos aparezcan dentro del notebook
import warnings
warnings.filterwarnings("ignore")  # oculta avisos para que la salida quede limpia

import pandas as pd              # DataFrame: la "hoja de cálculo" con la que trabajamos
import numpy as np               # cálculo numérico
import seaborn as sns            # gráficos estadísticos con poco código
import matplotlib.pyplot as plt  # base de todos los gráficos

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42  # semilla fija: así los resultados se pueden repetir

df = pd.read_csv("docs/workshop/data/mushrooms.csv")  # la tabla con todas las setas
print("Filas x columnas:", df.shape)
df.head()  # las 5 primeras filas

# Con texto, describe() da: count (filas), unique (valores distintos),
# top (el más repetido) y freq (cuántas veces sale).
df.describe()

# Tipo de cada columna: todas salen 'str' (texto; en pandas antiguos sale 'object').
# Cada letra es un código, p. ej. en 'class': e = comestible, p = venenosa.
df.info()

# ¿Cuántas setas de cada clase? (solo para conocer los datos, no para agrupar)
df["class"].value_counts()

# Contar nulos por columna. Sale todo 0... pero ojo: los valores desconocidos
# pueden estar escritos como '?' (lo veremos en la fase 02).
df.isnull().sum()

Filas x columnas: (8124, 23)
<class 'pandas.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   class                     8124 non-null   str  
 1   cap-shape                 8124 non-null   str  
 2   cap-surface               8124 non-null   str  
 3   cap-color                 8124 non-null   str  
 4   bruises                   8124 non-null   str  
 5   odor                      8124 non-null   str  
 6   gill-attachment           8124 non-null   str  
 7   gill-spacing              8124 non-null   str  
 8   gill-size                 8124 non-null   str  
 9   gill-color                8124 non-null   str  
 10  stalk-shape               8124 non-null   str  
 11  stalk-root                8124 non-null   str  
 12  stalk-surface-above-ring  8124 non-null   str  
 13  stalk-surface-below-ring  8124 non-null   str  
 14  stalk-color-above-ring

class                       0
cap-shape                   0
cap-surface                 0
cap-color                   0
bruises                     0
odor                        0
gill-attachment             0
gill-spacing                0
gill-size                   0
gill-color                  0
stalk-shape                 0
stalk-root                  0
stalk-surface-above-ring    0
stalk-surface-below-ring    0
stalk-color-above-ring      0
stalk-color-below-ring      0
veil-type                   0
veil-color                  0
ring-number                 0
ring-type                   0
spore-print-color           0
population                  0
habitat                     0
dtype: int64

## Fase 02 · Preparación de los datos

**Objetivo:** dejar los datos limpios y en números, que es lo único que entienden los modelos.

### Buscar valores extraños: ¿cuántos valores distintos tiene cada columna?

In [2]:
n_values = pd.DataFrame({
    "feature": df.columns,                                  # nombre de cada columna
    "n_values": [df[col].nunique() for col in df.columns],  # cuántos valores distintos tiene
})
n_values.sort_values("n_values").head()  # de menos a más: lo raro sale arriba

,feature,n_values
16,veil-type,1
0,class,2
4,bruises,2
6,gill-attachment,2
7,gill-spacing,2


Nadie nos dice dónde están los problemas: miramos **todos** los valores de **todas** las columnas.

In [3]:
# Imprimimos los valores distintos de cada columna, ordenados.
# Buscamos a ojo algo que no sea una letra-código: '?', '-', 'NA', ''...
for col in df.columns:
    print(f"{col:26} {sorted(df[col].unique())}")

class                      ['e', 'p']
cap-shape                  ['b', 'c', 'f', 'k', 's', 'x']
cap-surface                ['f', 'g', 's', 'y']
cap-color                  ['b', 'c', 'e', 'g', 'n', 'p', 'r', 'u', 'w', 'y']
bruises                    ['f', 't']
odor                       ['a', 'c', 'f', 'l', 'm', 'n', 'p', 's', 'y']
gill-attachment            ['a', 'f']
gill-spacing               ['c', 'w']
gill-size                  ['b', 'n']
gill-color                 ['b', 'e', 'g', 'h', 'k', 'n', 'o', 'p', 'r', 'u', 'w', 'y']
stalk-shape                ['e', 't']
stalk-root                 ['?', 'b', 'c', 'e', 'r']
stalk-surface-above-ring   ['f', 'k', 's', 'y']
stalk-surface-below-ring   ['f', 'k', 's', 'y']
stalk-color-above-ring     ['b', 'c', 'e', 'g', 'n', 'o', 'p', 'w', 'y']
stalk-color-below-ring     ['b', 'c', 'e', 'g', 'n', 'o', 'p', 'w', 'y']
veil-type                  ['p']
veil-color                 ['n', 'o', 'w', 'y']
ring-number                ['n', 'o', 't']
ring-typ

In [4]:
# Lo confirmamos contando: ¿cuántos '?' hay en cada columna? Nos quedamos con las que tengan alguno.
interrogantes = (df == "?").sum()
interrogantes[interrogantes > 0]

stalk-root    2480
dtype: int64

In [5]:
# Ya sabemos dónde mirar: ¿cuánto pesa el '?' dentro de 'stalk-root'?
df["stalk-root"].value_counts()

stalk-root
b    3776
?    2480
e    1120
c     556
r     192
Name: count, dtype: int64

**Hallazgos:**
- `veil-type` tiene **un único valor** → no aporta información.
- `stalk-root` tiene `'?'` en ~30 % de las filas → es un **nulo encubierto**. `isnull()` no lo ve porque para pandas `'?'` es un texto normal.

*(La documentación del dataset en UCI lo confirma: leer la documentación también forma parte del EDA).*

### Tratar los nulos encubiertos: `'?'` → moda

In [6]:
# La moda es el valor más repetido. La calculamos SIN contar los '?'.
moda = df.loc[df["stalk-root"] != "?", "stalk-root"].mode()[0]
print("Moda de stalk-root:", moda)

df["stalk-root"] = df["stalk-root"].replace("?", moda)  # cambiamos cada '?' por la moda
df["stalk-root"].value_counts()

Moda de stalk-root: b


stalk-root
b    6256
e    1120
c     556
r     192
Name: count, dtype: int64

### Quitar las columnas que no aportan

In [7]:
# Columnas con un solo valor: son iguales en todas las setas, no ayudan a distinguir nada
constantes = [col for col in df.columns if df[col].nunique() == 1]
print("Eliminamos:", constantes)
df = df.drop(columns=constantes)
df.shape

Eliminamos: ['veil-type']


(8124, 22)

### Separar la etiqueta (`y`) de las variables (`X`)

In [8]:
y = (df["class"] == "p").astype(int)  # 1 = venenosa, 0 = comestible (solo para validar)
X = df.drop(columns="class")          # las características de la seta
X.shape, y.value_counts().to_dict()

((8124, 21), {0: 4208, 1: 3916})

### Pasar las letras a números: One-Hot Encoding

Cada columna de letras se convierte en varias columnas 0/1, una por valor. Ejemplo: `odor` = `a`, `n`, `p`… → `odor_a`, `odor_n`, `odor_p`… Así no inventamos un orden falso (no es verdad que `a` < `n`).

In [9]:
X = pd.get_dummies(X, dtype=int)  # una columna 0/1 por cada valor posible
print("Columnas tras One-Hot:", X.shape[1])
X.iloc[:3, :8]  # vistazo a las primeras columnas

Columnas tras One-Hot: 115


,cap-shape_b,cap-shape_c,cap-shape_f,cap-shape_k,cap-shape_s,cap-shape_x,cap-surface_f,cap-surface_g
0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,1,0,0
2,1,0,0,0,0,0,0,0


### Separar entrenamiento y prueba (train / test)

Lo usaremos en la fase 03 para el Random Forest: se entrena con `train` y se examina con `test`, que son datos que no ha visto.

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=RANDOM_STATE)
X_train.shape, X_test.shape

((5443, 115), (2681, 115))

> **Resumen fase 02:** `'?'` → moda en `stalk-root`, fuera `veil-type`, etiqueta separada en `y` y **One-Hot** para pasar de 22 columnas de letras a columnas 0/1.